# Lesson 1.1 — Agentic Loops & stop_reason Control Flow

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain the agentic loop lifecycle: request, inspect stop_reason, execute tools, return results
- Drive loop continuation with stop_reason ('tool_use') and termination with 'end_turn'
- Append tool_result blocks to conversation history so the model can reason about the next action
- Identify and avoid agentic-loop anti-patterns called out on the CCAR-F exam

**You will produce:** A reusable agentic loop that terminates on stop_reason, not on parsed natural-language signals.

## Phase 1 Overview

Phase 1 covers **Domain 1 - Agentic Architecture & Orchestration (27%)**, the highest-weighted domain on CCAR-F. You will build the control structures that make autonomous Claude systems reliable: agentic loops, coordinator/subagent orchestration, subagent context passing, enforcement + handoff, lifecycle hooks, and session state.

By the end of this phase, you will have built:

- A `stop_reason`-driven agentic loop (Lesson 1.1)
- A hub-and-spoke coordinator/subagent design (Lesson 1.2)
- Explicit subagent context passing with the Task tool and AgentDefinition (Lesson 1.3)
- Programmatic enforcement gates and structured handoffs (Lesson 1.4)
- PostToolUse normalization and tool-call interception hooks (Lesson 1.5)
- Task-decomposition strategy plus session resume/fork (Lesson 1.6)

### Prerequisites

- Python 3 kernel with the `anthropic` SDK installed (`pip install anthropic`)
- An `ANTHROPIC_API_KEY` for live calls (notebooks also run offline with mock fallbacks)
- 6+ months hands-on with the Claude Messages API, Agent SDK, Claude Code, and MCP

## Concepts

An **agentic loop** repeatedly sends the conversation to Claude, inspects the response's `stop_reason`, and acts on it:

- `stop_reason == "tool_use"` -> the model wants to call one or more tools. Execute them, append the results as a `user` message of `tool_result` blocks, and loop again.
- `stop_reason == "end_turn"` -> the model is done. Stop and surface the final text.

Every tool result must be appended to the conversation history so the model can incorporate new information into its next decision. This is what makes the loop *model-driven* (Claude reasons about which tool to call next from context) rather than a hard-coded decision tree.

### Exam anti-patterns (do NOT do these)
- **Parsing natural-language signals** (e.g. looking for the word "done") to decide when to stop. The authoritative signal is `stop_reason`.
- **Arbitrary iteration caps as the *primary* stopping mechanism.** A cap is a safety backstop, not the termination logic.
- **Checking for assistant text content** as a completion indicator - a turn can contain both text and `tool_use` blocks.

> **Reference:** [Anthropic - Tool use & the agentic loop](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define a tool and inspect the first stop_reason

We declare one tool with a JSON-schema `input_schema`. On the first turn Claude should return `stop_reason == "tool_use"`. We read it rather than scanning the text.

In [ ]:
TOOLS = [{
    "name": "get_weather",
    "description": "Get the current temperature for a city. Input: a city name string.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

def run_tool(name, args):
    if name == "get_weather":
        return f"18C and clear in {args['city']}"
    return f"[unknown tool {name}]"

messages = [{"role": "user", "content": "What's the weather in Paris?"}]

if HAVE_API:
    resp = client.messages.create(model=MODEL_HAIKU, max_tokens=512,
                                   tools=TOOLS, messages=messages)
    print("stop_reason:", resp.stop_reason)
else:
    print("stop_reason:", "tool_use", "(mock)")

## Code-Along: Step 2 — A correct stop_reason-driven loop

The loop continues **only** while `stop_reason == "tool_use"`. We append the assistant turn and a `tool_result` user turn each iteration. `max_iters` is a safety backstop - never the main stopping condition.

In [ ]:
def agentic_loop(user_text, max_iters=10):
    messages = [{"role": "user", "content": user_text}]
    if not HAVE_API:
        return mock_text(user_text)
    for _ in range(max_iters):  # backstop only
        resp = client.messages.create(model=MODEL_HAIKU, max_tokens=1024,
                                       tools=TOOLS, messages=messages)
        if resp.stop_reason != "tool_use":
            # end_turn (or other) -> finished
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = run_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": out,
                })
        messages.append({"role": "user", "content": tool_results})
    return "[stopped: hit max_iters backstop]"

print(agentic_loop("What's the weather in Paris?"))

## Code-Along: Step 3 — Contrast with the anti-pattern

This version parses assistant text for the word "done" to terminate. It is fragile: the model may emit text alongside a `tool_use` block, or never say "done". Study it so you can recognize and reject it on the exam.

In [ ]:
def bad_loop(messages):
    # ANTI-PATTERN: natural-language termination + text-content completion check
    while True:
        resp = call_model(messages)                 # pseudo
        text = get_text(resp)                        # pseudo
        if "done" in text.lower():                 # <- fragile NL parsing
            break
        # also wrongly treats ANY text as completion
    return text

print("Reject bad_loop: terminate on stop_reason == 'end_turn', not parsed text.")

## Exercise

Add a second tool `get_time(city)` and extend `run_tool`. Ask a multi-tool question ("What's the weather and time in Paris?") and confirm your loop issues multiple iterations driven purely by `stop_reason`. Then add an assertion that fails if the loop ever terminates by inspecting text content.

In [ ]:
# TODO: add get_time to TOOLS with an input_schema
# TODO: extend run_tool to handle get_time
# TODO: call agentic_loop with a two-part question and print the result
# TODO: assert that termination depends only on stop_reason (not on text parsing)

## Key Takeaways

- `stop_reason == 'tool_use'` means continue; `'end_turn'` means stop.
- Append tool_result blocks to history every iteration so the model reasons over fresh data.
- Iteration caps are a safety backstop, never the primary termination mechanism.
- Never parse natural language or treat text content as a completion signal.

---

**Next:** [Lesson 1.2 — Coordinator / Subagent Orchestration](./lesson-1.2-coordinator-subagent.ipynb)